# SSCOFS -- experiments with subsetting

This is working with looking at the raw netcdf files on the NODD.

In [1]:
from pathlib import Path
import time
import netCDF4
import requests
import xarray as xr
import xarray_subset_grid as xsg


In [2]:
# sample file.
# auto downloaded from:
# https://noaa-nos-ofs-pds.s3.amazonaws.com/index.html#sscofs/netcdf/

base_url = "https://noaa-nos-ofs-pds.s3.amazonaws.com/sscofs/netcdf/2026/11/11/"
infiles = ["sscofs.t03z.20261111.fields.f000.nc", # note: MFDataset didn'twork out of the box
          ]
          #  "sscofs.t03z.20261111.fields.f001.nc",
          #  "sscofs.t03z.20261111.fields.f002.nc",
          #  "sscofs.t03z.20261111.fields.f003.nc",
          # ]

for infile in infiles:
    if Path(infile).is_file():
        print(infile, "already there")
    else:
        print("downloading:", infile)
        response = requests.get(base_url + infile)
        response.raise_for_status()
    
        with open(infile, "wb") as file:
            file.write(response.content)
        if not Path(infile).is_file():
            print(infile, "Didn't download !!! ")


sscofs.t03z.20261111.fields.f000.nc already there


In [3]:
# ds = xr.open_mfdataset(infiles, concat_dim='time', combine='nested')
ds = xr.open_dataset(infiles[0])


In [4]:
ds

<xarray.Dataset> Size: 199MB
Dimensions:             (nele: 433410, node: 239734, siglev: 11, three: 3,
                         time: 1, maxnode: 11, maxelem: 9, four: 4, siglay: 10)
Coordinates:
    lonc                (nele) float32 2MB ...
    latc                (nele) float32 2MB ...
    lon                 (node) float32 959kB ...
    lat                 (node) float32 959kB ...
    siglev              (siglev, node) float32 11MB ...
  * time                (time) datetime64[ns] 8B 2026-11-11T03:00:00
    siglay              (siglay, node) float32 10MB ...
Dimensions without coordinates: nele, node, three, maxnode, maxelem, four
Data variables: (12/44)
    nprocs              int32 4B ...
    partition           (nele) int32 2MB ...
    x                   (node) float32 959kB ...
    y                   (node) float32 959kB ...
    xc                  (nele) float32 2MB ...
    yc                  (nele) float32 2MB ...
    ...                  ...
    wet_nodes           (time, node) int32 959kB ...
    wet_cells           (time, nele) int32 2MB ...
    wet_nodes_prev_int  (time, node) int32 959kB ...
    wet_cells_prev_int  (time, nele) int32 2MB ...
    wet_cells_prev_ext  (time, nele) int32 2MB ...
    inundation_cells    (time, nele) int32 2MB ...
Attributes: (12/14)
    title:                       SSCOFS
    institution:                 School for Marine Science and Technology
    source:                      FVCOM_4.4.7
    history:                     model started at: 11/11/2026   04:10
    references:                  http://fvcom.smast.umassd.edu, https://githu...
    Conventions:                 CF-1.0
    ...                          ...
    Tidal_Forcing:               Tidal Forcing Time Series Title: sscofs late...
    River_Forcing:               THERE ARE 38 RIVERS IN THIS MODEL.\nRIVER IN...
    GroundWater_Forcing:         GROUND WATER FORCING IS OFF!
    Surface_Heat_Forcing:        FVCOM variable surface heat forcing file:\nF...
    Surface_Wind_Forcing:        FVCOM variable surface Wind forcing:\nFILE N...
    Surface_PrecipEvap_Forcing:  SURFACE PRECIPITATION FORCING IS OFF

# Try subsetting

In [5]:
# small
# BBox = (-125.3, 44.29, -123.8, 45.11)
# larger
BBox = (-122, 47.5, -123.5, 48.1)


In [6]:
# info on chunkcache sizes:
# See: https://docs.unidata.ucar.edu/netcdf-c/current/nc4cache_8c.html
# in short:
# size: =64 * 1024 * 1024, nelems=1009, preemption=0.75
# Check current global default: returns (size_bytes, nelems, preemption)
size, nelems, preemption = netCDF4.get_chunk_cache()
print(f"Size: {size / (1024**2):.1f} MB, Slots: {nelems}, Preemption: {preemption}")

# Set global default (e.g., 64 MB cache, 1009 hash slots, 0.75 preemption)
# netCDF4.set_chunk_cache(size=64 * 1024 * 1024, nelems=1009, preemption=0.75)

Size: 64.0 MB, Slots: 1000, Preemption: 0.75


In [7]:
# need to add the topology for subsetting to work
ds = xsg.grids.ugrid.assign_ugrid_topology(ds, face_node_connectivity='nv')

In [8]:
# the mesh variable
print(ds['mesh'])

<xarray.DataArray 'mesh' ()> Size: 4B
array(0, dtype=int32)
Attributes:
    cf_role:                 mesh_topology
    topology_dimension:      2
    face_node_connectivity:  nv
    face_face_connectivity:  nbe
    node_coordinates:        lon lat
    face_coordinates:        lonc latc
    face_dimension:          nele
    start_index:             1


In [9]:
ds2 = ds.xsg.subset_vars(['u', 'v', 'temp', 'salinity'])

In [10]:
start = time.time()
ds3 = ds2.xsg.subset_bbox(BBox)
print(f"it took {time.time() - start} seconds to compute the subset")

it took 37.9035108089447 seconds to compute the subset


In [11]:
start = time.time()
ds3.to_netcdf("sscofs_subset.nc")
print(f"With chunk_cache={size / (1024**2):.1f}MB it took {(time.time() - start) / 60} minutes to write nc file")

With chunk_cache=64.0MB it took 3.9876280546188356 minutes to write nc file


In [12]:
print(f"With chunk_cache={size / (1024**2):.1f}MB it took {(time.time() - start) / 60} minutes to compute the subset")

With chunk_cache=64.0MB it took 3.987862753868103 minutes to compute the subset


In [13]:
# Set global default (e.g., 64 MB cache, 1009 hash slots, 0.75 preemption)
netCDF4.set_chunk_cache(size=1024 * 1024 * 1024, nelems=1009, preemption=0.75)

size, nelems, preemption = netCDF4.get_chunk_cache()
print(f"Size: {size / (1024**2):.1f} MB, Slots: {nelems}, Preemption: {preemption}")


Size: 1024.0 MB, Slots: 1009, Preemption: 0.75


In [14]:
ds = xr.open_dataset(infiles[0])
ds = xsg.grids.ugrid.assign_ugrid_topology(ds, face_node_connectivity='nv')
ds2 = ds.xsg.subset_vars(['u', 'v', 'temp', 'salinity'])


In [ ]:
start = time.time()
ds3.to_netcdf("sscofs_subset.nc")
print(f"With chunk_cache={size / (1024**2):.1f}MB it took {(time.time() - start) / 60} minutes to compute the subset")

In [ ]:
start = time.time()
ds3.to_netcdf("sscofs_subset.nc")
print(f"With chunk_cache={size / (1024**2):.1f}MB it took {(time.time() - start) / 60} minutes to write nc file")